# G1 Docket — Interactive Quickstart & Developer Walkthrough

This notebook demonstrates how to interact with **G1 Docket** using the Python SDK (`docket`):
1. **Connecting to the Cluster & Health Inspection**
2. **Verifying Active Snapshot Attestation Records**
3. **AI Inference Protection (< 1 ms in-memory admission control)**
4. **Tool-Calling Protection with `@attested_tool`**
5. **RAG Pipeline Integration (LangChain & LlamaIndex)**
6. **Attribute-Based Access Control (ABAC) & Data Residency**
7. **Pre-Dispatch Prompt Auditing with LangChain Callback Handler**

## 1. Connect & Verify Cluster Health
We initialize `DocketClient` pointing to our local or remote cluster.

In [ ]:
from docket import DocketClient

client = DocketClient(base_url="http://localhost:18082")
health = client.get_health()
print(f"[+] Prevalidator Status: {health.get('status')} | Mode: {health.get('mode')} | Active Snapshot Sequence: {health.get('snapshot_sequence')}")

## 2. Inspect In-Memory Snapshot Contracts
Look up attestation records directly from the Prevalidator or Authoritative Registry.

In [ ]:
snapshot = client.get_latest_snapshot()
print(f"Snapshot Payload Type: {snapshot.get('payloadType')}")
print(f"Signatures present: {len(snapshot.get('signatures', []))}")

## 3. Inline Protection & Admission Gating
When prompts carry data wrapped in `<pod:object>` tags, the Prevalidator evaluates the caller key, content digest, and allowed egress routes in < 1 ms.

In [ ]:
from docket import PolicyDenialError

# A. Authorized Public Egress
public_prompt = (
    "Please summarize this verified document:\n"
    '<pod:object locator="file://estate/fileshare/product_overview_public.md">'
    "Open Source AI Architecture Overview"
    "</pod:object>"
)

resp = client.chat_completions(
    messages=[{"role": "user", "content": public_prompt}],
    model="gpt-4o",
    caller_key="pod-key-public",
)
print("Public Request Allowed:", resp["choices"][0]["message"]["content"])

# B. Unauthorized Sensitive Egress (Snowflake Restricted Table to Public LLM)
restricted_prompt = (
    "Analyze customer contacts:\n"
    '<pod:object locator="snowflake://contoso.us-east-1.snowflakecomputing.com/FINANCE_PROD/PUBLIC/DIM_CUSTOMER_CONTACTS">'
    "CUST-901 | elena.fisher@example.com | phone: 312-555-0199"
    "</pod:object>"
)

try:
    client.chat_completions(
        messages=[{"role": "user", "content": restricted_prompt}],
        model="gpt-4o",
        caller_key="pod-key-public",
    )
except PolicyDenialError as err:
    print(f"\n[!] Expected G1 Docket Refusal (HTTP {err.status_code}): {err.response_body.get('error', {}).get('message')}")

## 4. Protecting Agent Tools with `@attested_tool`
Decorate any tool function to automatically wrap its outputs in verified `<pod:object>` tokens with automatic SHA-256 calculation.

In [ ]:
from docket import attested_tool

@attested_tool("snowflake://analytics.corp.com/FINANCE/PUBLIC/Q3_REVENUE", auto_hash=True)
def get_sales_kpi(region: str) -> str:
    return f"{region} Revenue: \$18.4M | Growth: +24%"

tool_output = get_sales_kpi("EMEA")
print("Attested Tool Output:")
print(tool_output)

## 5. RAG Pipeline Transformers (LangChain & LlamaIndex)
Use drop-in pipeline components to automatically wrap retrieved documents or nodes.

In [ ]:
from docket import DocketDocumentTransformer, DocketLlamaIndexPostprocessor

# LangChain Document Transformer
class MockLangChainDoc:
    def __init__(self, content, source):
        self.page_content = content
        self.metadata = {"source": source}

transformer = DocketDocumentTransformer(auto_hash=True)
raw_docs = [MockLangChainDoc("Enterprise Architecture Specs", "s3://lake/specs.md")]
attested_docs = transformer.transform_documents(raw_docs)
print("LangChain Transformed Page Content:")
print(attested_docs[0].page_content)

## 6. Attribute-Based Access Control (ABAC) & Data Residency
Pass departmental claims and sovereign target regions to enforce zero-trust compartmentalization.

In [ ]:
# Cleared Finance Agent accessing Finance data
finance_prompt = (
    "Summarize verified financial context:\n"
    '<pod:object locator="file://estate/fileshare/product_overview_public.md">'
    "Open Source AI Architecture Overview"
    "</pod:object>"
)

approved_resp = client.chat_completions(
    messages=[{"role": "user", "content": finance_prompt}],
    model="gpt-4o",
    caller_key="pod-key-confidential",
    caller_dept="Finance",
    target_region="us-east-1",
)
print("Cleared Finance Prompt Response:", approved_resp["choices"][0]["message"]["content"])

## 7. Pre-Dispatch Prompt Auditing with LangChain Callback Handler
Enforce data attestation directly before prompt dispatch to an LLM, raising a 403 `PolicyDenialError` if unpapered data is used.

In [ ]:
from docket.integrations import DocketLangChainCallbackHandler
from docket import PolicyDenialError

handler = DocketLangChainCallbackHandler(require_attested_context=True, max_untagged_chars=100)

# 1. Allowed prompt with <pod:object> tag
valid_prompt = 'Analyze: <pod:object locator="s3://lake/specs.md">Attested Specs</pod:object>'
handler.on_llm_start(serialized={}, prompts=[valid_prompt])
print("[+] Valid Prompt Passed Callback Audit. Decision:", handler.audit_log[-1]["decision"])

# 2. Blocked prompt lacking required attestation tags
try:
    handler.on_llm_start(serialized={}, prompts=["Unpapered context without tags"])
except PolicyDenialError as err:
    print(f"[!] Prompt Blocked by Callback Handler (HTTP {err.status_code}):", err.response_body)